# Benchmark OCR sur GPU Kaggle

Seule la **Phase 1** (inference OCR) a besoin d'un GPU. Les evaluations
(TEDS, fusion, champs) tournent en quelques secondes sur ton PC : on les
garde en local, on ne rapatrie que `results/`.

## Avant de lancer

1. **Settings -> Accelerator -> GPU T4 x2** (ou P100)
2. **Settings -> Internet -> On** (sinon ni `pip install` ni `git clone`)
3. **Add Input -> Datasets** : ajoute ton dataset d'images
4. Verifie que le depot GitHub est a jour (`git push` depuis ton PC)

Puis **Save Version -> Save & Run All (Commit)** : le notebook tourne en
arriere-plan, tu peux fermer l'onglet. Sinon la session meurt avec la page.

## 1. Verifier le GPU

In [ ]:
!nvidia-smi
import sys, os
print(sys.version)

## 2. Recuperer le code

Le depot ne contient que le code : les images vivent dans le dataset
Kaggle, et `results/` est ignore par git.

**Si le depot est prive**, ajoute un jeton GitHub dans *Add-ons ->
Secrets* sous le nom `GITHUB_TOKEN` (jeton a creer sur GitHub :
*Settings -> Developer settings -> Personal access tokens -> Fine-grained*,
acces en lecture au seul depot PTITOcr). La cellule le detecte toute
seule.

`GIT_TERMINAL_PROMPT=0` est indispensable : sans lui, un depot prive fait
attendre `git clone` sur une invite de mot de passe que personne ne lira,
et le notebook reste bloque sans le moindre message d'erreur.

In [ ]:
import os, subprocess

# Jamais d'invite interactive : on veut une erreur, pas un blocage.
os.environ["GIT_TERMINAL_PROMPT"] = "0"

OWNER, REPO, SECRET = "Halfaa", "PTITOcr", "GITHUB_TOKEN"

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret(SECRET)
    print(f"Secret '{SECRET}' trouve ({len(token)} caracteres)")
except ImportError:
    print("kaggle_secrets indisponible : tu n'es pas sur Kaggle ?")
except Exception as e:
    # On affiche la vraie cause. Le plus frequent n'est pas l'absence du
    # secret mais son NON-RATTACHEMENT au notebook courant : Kaggle les
    # stocke par compte, et chaque notebook doit cocher ceux qu'il utilise.
    print(f"Secret '{SECRET}' illisible -> {type(e).__name__}: {e}")
    print("\nA verifier, dans cet ordre :")
    print("  1. Add-ons -> Secrets : le secret est-il RATTACHE a CE notebook")
    print("     (interrupteur a droite de son nom) ? C'est l'oubli habituel.")
    print(f"  2. Le label est-il exactement '{SECRET}' ? Casse comprise,")
    print("     sans espace avant ou apres.")
    print("  3. Le secret a-t-il ete cree APRES le demarrage de la session ?")
    print("     Si oui : Run -> Restart Session, puis rejoue cette cellule.")

url = (f"https://{token}@github.com/{OWNER}/{REPO}.git" if token
       else f"https://github.com/{OWNER}/{REPO}.git")

%cd /kaggle/working
!rm -rf {REPO}
r = subprocess.run(["git", "clone", "--depth", "1", url, REPO],
                   capture_output=True, text=True, timeout=180)
if r.returncode != 0:
    # On masque le jeton avant tout affichage : la sortie d'un notebook
    # Kaggle est conservee dans la version publiee.
    msg = r.stderr.replace(token, "***") if token else r.stderr
    raise SystemExit("Clone impossible.\n" + msg)

%cd /kaggle/working/{REPO}
!ls

## 3. Localiser les images du dataset

Kaggle monte tes datasets en lecture seule sous `/kaggle/input/<nom>/`.
Cette cellule trouve le dossier toute seule plutot que de coder le chemin
en dur, qui change des que tu renommes le dataset.

In [ ]:
from pathlib import Path

# Le dossier d'images = celui qui contient le plus de page_*.png
candidates = {}
for png in Path("/kaggle/input").rglob("page_*.png"):
    candidates[png.parent] = candidates.get(png.parent, 0) + 1

for folder, n in sorted(candidates.items(), key=lambda kv: -kv[1]):
    print(f"{n:>4} images   {folder}")

assert candidates, "Aucune image trouvee : as-tu ajoute le dataset via Add Input ?"
IMAGES_DIR = max(candidates, key=candidates.get)
print("\n-> retenu :", IMAGES_DIR)

## 4. Installer les dependances

**numpy < 2 est imperatif** : paddlex ne supporte pas NumPy 2.x, et l'erreur
qu'il leve ("a dependency error occurred during pipeline creation") ne nomme
ni le paquet ni l'exception reelle. `src/env_check.py` verifie ce plafond au
demarrage et echoue en une seconde plutot qu'apres le chargement du modele.

Redemarre le noyau apres cette cellule si numpy etait deja importe.

In [ ]:
import subprocess, sys, re, importlib

def pip(*args):
    """pip qui ECHOUE bruyamment.

    `!pip install` dans un notebook n'interrompt pas la cellule quand il
    rate : le run continue et ne s'effondre que bien plus tard, sur un
    ModuleNotFoundError qui ne dit pas d'ou vient le probleme.
    """
    r = subprocess.run([sys.executable, "-m", "pip", "install", *args],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("\n".join(r.stdout.splitlines()[-5:]))
        print("\n".join(r.stderr.splitlines()[-8:]))
    return r.returncode == 0

assert pip("-q", "numpy<2", "opencv-python<4.12"), "installation de numpy/opencv echouee"

# paddlepaddle-gpu N'EST PAS sur PyPI : il faut l'index de PaddlePaddle.
#
# nvidia-smi annonce la version CUDA MAXIMALE que supporte le pilote, pas
# celle qu'il faut demander. Kaggle affiche 13.0 alors que Paddle ne publie
# pas d'index cu130 : les roues cu126 tournent tres bien sur ce pilote, CUDA
# etant retro-compatible. On essaie donc les index du plus recent au plus
# ancien plutot que d'en deduire un seul, forcement fragile.
BASE = "https://www.paddlepaddle.org.cn/packages/stable"

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
m = re.search(r"CUDA Version:\s*(\d+)\.(\d+)", smi)
major = int(m.group(1)) if m else 12
tags = ["cu126", "cu123", "cu118"] if major >= 12 else ["cu118"]
print(f"Pilote CUDA {m.group(0) if m else '?'} -> essais : {tags}")

for tag in tags:
    print(f"\n--- {tag}")
    if not pip("-q", "paddlepaddle-gpu", "-i", f"{BASE}/{tag}/"):
        continue
    try:
        importlib.invalidate_caches()
        import paddle
        print(f"paddle {paddle.__version__} installe depuis {tag}")
        break
    except Exception as e:
        print(f"import paddle impossible apres {tag} : {e}")
else:
    raise SystemExit(
        "Aucun index Paddle n'a fonctionne. Verifie les index disponibles sur\n"
        "https://www.paddlepaddle.org.cn/en/install/quick (l'ensemble change\n"
        "d'une version a l'autre).")

assert pip("-q", "paddleocr[doc-parser]>=3.4.0"), "installation de paddleocr echouee"

# Restauration de NCCL, EN DERNIER.
#
# Kaggle preinstalle torch, et paddleocr l'importe. Or les dependances de
# paddlepaddle-gpu retrogradent nvidia-nccl-cu12, apres quoi le torch de
# Kaggle ne trouve plus un symbole qu'il attend :
#     libtorch_cuda.so: undefined symbol: ncclCommShrink
# torch >= 2.1 exige NCCL >= 2.18. On le remet donc a jour une fois tous
# les autres paquets poses, sinon un install ulterieur le redegraderait.
assert pip("-q", "-U", "nvidia-nccl-cu12"), "restauration de NCCL echouee"

import importlib.metadata as md
for pkg in ("numpy", "paddlepaddle-gpu", "paddleocr", "paddlex",
            "torch", "nvidia-nccl-cu12"):
    try:
        print(f"{pkg:<20}{md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:<20}absent")

In [ ]:
# Verification des DEUX piles avant de lancer une heure de calcul.
# paddleocr importe torch : un torch casse fait echouer le benchmark, et
# le message qu'on recolte alors ne parle meme pas de torch.
import subprocess, sys

check = subprocess.run([sys.executable, "-c", """
import torch, paddle
print('torch  ', torch.__version__, '| cuda:', torch.cuda.is_available())
print('paddle ', paddle.__version__, '| cuda:', paddle.is_compiled_with_cuda(),
      '|', paddle.device.cuda.device_count(), 'GPU')
assert torch.cuda.is_available(), 'torch ne voit pas le GPU'
assert paddle.is_compiled_with_cuda(), 'roue Paddle sans CUDA'
assert paddle.device.cuda.device_count() > 0, 'aucun GPU visible'
"""], capture_output=True, text=True)

print(check.stdout)
if check.returncode != 0:
    print(check.stderr)
    raise SystemExit(
        "Environnement inutilisable -- ne lance pas le benchmark.\n"
        "Si l'erreur mentionne ncclCommShrink ou libtorch_cuda.so, c'est que\n"
        "NCCL a ete redegrade : rejoue la cellule d'installation.\n"
        "Si elle mentionne le GPU : Settings -> Accelerator -> GPU T4 x2.")
print("Environnement OK.")

## 4bis. Pretraitement (optionnel)

Le depot est deja clone, donc `src/preprocess.py` est disponible ici : on
peut pretraiter sur place au lieu de televerser un second dataset. Deux
avantages concrets.

Le code de pretraitement est **toujours celui du depot**. Des images
pretraitees televersees se perimeraient a la premiere modification du
module, sans que rien ne le signale -- on mesurerait alors une version du
pretraitement qui n'existe plus.

Et les DEUX lectures sortent de la meme session, ce dont la politique
adaptative de `src/adaptive.py` a besoin : elle choisit entre lecture brute
et lecture pretraitee, donc il lui faut les deux.

In [ ]:
PASSES = ["raw", "preprocessed"]   # mettre ["raw"] pour n'en faire qu'une

import subprocess, sys
from pathlib import Path

dossiers = {"raw": str(IMAGES_DIR)}

if "preprocessed" in PASSES:
    sortie = Path("/kaggle/working/preprocessed")
    r = subprocess.run(
        [sys.executable, "src/preprocess.py",
         "--images-dir", str(IMAGES_DIR), "--out", str(sortie), "--debug"],
        cwd="/kaggle/working/PTITOcr", capture_output=True, text=True)
    print("\n".join(r.stdout.splitlines()[-8:]))
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise SystemExit("Le pretraitement a echoue.")

    n = len(list(sortie.glob("page_*.png")))
    attendu = len(list(Path(IMAGES_DIR).glob("page_*.png")))
    assert n == attendu, f"{n} images pretraitees pour {attendu} attendues"
    dossiers["preprocessed"] = str(sortie)

for nom, chemin in dossiers.items():
    print(f"  {nom:<14}{chemin}")

## 5. Lancer le benchmark

Le script ecrit dans `results/<--output>/` et reecrit son rapport apres
chaque image : une interruption ne coute qu'une page. Relancer reutilise
les pages deja faites.

In [ ]:
MODEL = "paddleocr-vl"          # paddleocr-vl | ppstructure | hunyuan
PREFIXE = "variants_warped"     # les runs s'appelleront <prefixe>_<passe>_<modele>

import subprocess, sys

runs = {}
for passe, chemin in dossiers.items():
    run = f"{PREFIXE}_{passe}_{MODEL.replace('-', '')}"
    runs[passe] = run
    print(f"\n{'=' * 60}\n{passe} -> {run}\n{'=' * 60}", flush=True)
    # Le script reecrit son rapport apres chaque image : une interruption
    # ne coute qu'une page, et relancer reutilise ce qui est deja fait.
    subprocess.run(
        [sys.executable, "scripts/bench_deformed.py", "--model", MODEL,
         "--images-dir", chemin, "--output", run],
        cwd="/kaggle/working/PTITOcr", check=True)

print("\nRuns produits :", list(runs.values()))

## 6. Rapatrier les resultats

Tout ce qui se trouve dans `/kaggle/working` est telechargeable depuis
l'onglet **Output** a la fin du run. On zippe pour n'avoir qu'un fichier.

Puis, **sur ton PC**, decompresse dans `MedicalRecord/results/` et lance :

```powershell
python src\eval_teds.py            --run multipage_preproc2_paddleocr
python src\eval_multipage_merge.py --run multipage_preproc2_paddleocr
python src\eval_fields.py --runs multipage_paddleocr_vl multipage_preproc2_paddleocr
```

In [ ]:
import shutil
from pathlib import Path

base = Path("/kaggle/working/PTITOcr/results")
for passe, run in runs.items():
    src = base / run
    assert src.exists(), f"{src} absent : le benchmark a-t-il tourne ?"
    out = shutil.make_archive(f"/kaggle/working/{run}", "zip", base, run)
    print(f"{Path(out).name:<45}{Path(out).stat().st_size / 1e6:>6.1f} Mo")

# Le depot clone pese pour rien dans l'Output : on le retire.
shutil.rmtree("/kaggle/working/PTITOcr", ignore_errors=True)
shutil.rmtree("/kaggle/working/preprocessed", ignore_errors=True)
print("\nContenu de /kaggle/working :")
for f in sorted(Path("/kaggle/working").iterdir()):
    print(" ", f.name)

---
## Variante HunyuanOCR

Environnement incompatible avec Paddle (transformers >= 5.13) : fais-en un
notebook separe plutot que de melanger les deux dans la meme session.

**Attention licence.** La licence communautaire Tencent exclut l'Union
europeenne, le Royaume-Uni et la Coree du Sud de son Territoire, et Kaggle
ne permet pas de choisir la region de la machine. PaddleOCR-VL, en Apache
2.0, n'a aucune de ces contraintes. A trancher avec ton encadrante avant de
lancer Hunyuan ici.

In [ ]:
# !pip install -q "transformers>=5.13.0" accelerate pillow
# !python scripts/bench_deformed.py --model hunyuan \
#     --images-dir "$IMAGES_DIR" --output multipage_preproc2_hunyuan